<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2FACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/ACSM_Track1_Serverless_Seamless_Data_Ingestion.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

---

# Track 1 (Notebook 1): Unified Bronze Lakehouse & Multi-Cloud Data Access in BigQuery Studio UI
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ 1. Unified Bronze Lakehouse Architecture (3 Storage Engines in 1 BigQuery UI)

Before building our Silver & Gold Medallion pipelines in **Notebook 2**, this notebook establishes **BigQuery Studio as ACSM's Single Unified Data Access Layer (`RFP Clauses C1.1.1.1, C1.1.1.2, C1.1.1.3, C1.1.1.4 & C1.1.1.18`)** across **three storage engines**:
1. **BigQuery Native Storage (`6` Fact Tables in `acsm_bronze` — `1,233,284` rows)**: `Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, and `Fact_CC_Collection` ingested via serverless `LOAD DATA OVERWRITE` at **`$0` / `0 Bytes Billed`**.
2. **GCP Lakehouse Open-Source Apache Iceberg Table (`acsm_bronze.m3CIF` — `100,000` rows)**: Customer CIF Master created as a **BigLake Managed Apache Iceberg Table (`table_format = 'ICEBERG'`, `file_format = 'PARQUET'`)** stored on Google Cloud Storage (`gs://.../iceberg/m3CIF`) with `EXPORT TABLE METADATA`, **Schema Evolution (`ADD COLUMN`)**, and **Time Travel (`FOR SYSTEM_TIME AS OF`)**.
3. **Cross-Cloud AWS Glue Federated Apache Iceberg Table (`acsm_aws_federated_catalog.acsm_aws_bronze.dimProduct` — `65,000` rows)**: Credit Card Product Master stored in **Amazon S3 (`s3://...`) + AWS Glue Data Catalog (`<AWS_ACCOUNT_ID>` / `acsm-gcp-trust-role` / `acsm-federated-only-policy`)** and queried directly from BigQuery Studio without cross-cloud ETL!

![Track 1 Notebook 1 — Unified Bronze Lakehouse Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook1_serverless_ingestion_flow.png)

---

## 🔗 2. Cross-Cloud AWS Glue Catalog Federation Set Up (`acsm_aws_bronze.dimProduct`)

![Cross-Cloud AWS Glue Catalog Federation Set Up — Slide 6](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/slide6_aws_glue_flow.png)

| Storage Engine in Bronze | ACSM Table(s) | Row Count | Storage Format & Location |
| :--- | :--- | :---: | :--- |
| **1. BigQuery Native Storage** | `Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection` | `1,233,284` rows | BigQuery Capacitor Native Storage (`asia-southeast1`) |
| **2. GCP Lakehouse Apache Iceberg** | **`acsm_bronze.m3CIF`** *(Customer CIF Master)* | `100,000` rows | Open Apache Parquet + Iceberg `v*.metadata.json` (`gs://acsm-workshop-landing-${PROJECT_ID}/iceberg/m3CIF`) |
| **3. AWS Glue Federated Apache Iceberg** | **`acsm_aws_bronze.dimProduct`** *(Credit Card Product Master)* | `65,000` rows | Amazon S3 Apache Parquet + AWS Glue Catalog (`s3://.../acsm_aws_bronze/dimProduct/` in `ap-southeast-1`) |


---
## Step 0 (Prerequisite): Create VPC Network & Singapore (`asia-southeast1`) Subnetwork in Google Cloud Shell First

> **⚠️ IMPORTANT — RUN ONCE IN GOOGLE CLOUD SHELL BEFORE CONNECTING THIS NOTEBOOK**
> BigQuery Studio Notebooks (powered by Colab Enterprise) require a **VPC Network** and a **Regional Subnetwork in Singapore (`asia-southeast1`)** with **Private Google Access** enabled to provision the notebook runtime.
> 1. Click **Activate Cloud Shell (`>_`)** in the top-right corner of the Google Cloud Console.
> 2. Copy and run the `gcloud` commands below in **Cloud Shell**.
> 3. Once complete, come back to this Notebook in **BigQuery Studio**, click **Connect** (top-right), and select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`** (`asia-southeast1`).

```bash
# =============================================================================
# Run these commands in Google Cloud Shell (>_) BEFORE connecting the notebook
# =============================================================================
export PROJECT_ID="<YOUR_GCP_PROJECT_ID>"   # e.g., export PROJECT_ID="${PROJECT_ID}"
export LOCATION="asia-southeast1"           # Always Singapore (asia-southeast1)
export NETWORK_NAME="acsm-colab-network"
export SUBNET_NAME="acsm-colab-subnet-sg"

gcloud config set project "${PROJECT_ID}"

# 1. Enable required APIs for BigQuery Studio Notebooks (Colab Enterprise)
gcloud services enable \
  bigquery.googleapis.com \
  aiplatform.googleapis.com \
  compute.googleapis.com \
  dataform.googleapis.com \
  --project="${PROJECT_ID}"

# 2. Create Custom VPC Network
gcloud compute networks create "${NETWORK_NAME}" \
  --project="${PROJECT_ID}" \
  --subnet-mode=custom

# 3. Create Regional Subnetwork in Singapore (asia-southeast1) with Private Google Access
gcloud compute networks subnets create "${SUBNET_NAME}" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}" \
  --range="10.10.0.0/24" \
  --enable-private-ip-google-access

# 4. Create Cloud Router & Cloud NAT in Singapore (allows git clone from GitHub without public IPs)
gcloud compute routers create "acsm-colab-router-sg" \
  --project="${PROJECT_ID}" \
  --network="${NETWORK_NAME}" \
  --region="${LOCATION}"

gcloud compute routers nats create "acsm-colab-nat-sg" \
  --project="${PROJECT_ID}" \
  --router="acsm-colab-router-sg" \
  --region="${LOCATION}" \
  --auto-allocate-nat-external-ips \
  --nat-all-subnet-ip-ranges
```

> **🔍 How to Verify Step 0 on GCP Console UI & Connect the Notebook Runtime**
> 1. Open **VPC network $\rightarrow$ VPC networks** in the GCP Console and verify **`acsm-colab-network`** and subnet **`acsm-colab-subnet-sg`** (`Region: asia-southeast1`, `Private Google access: On`) are listed.
> 2. Come back to **BigQuery Studio**, open this notebook, click the **Connect** dropdown (top-right) $\rightarrow$ **Connect to a runtime** (or **Create a runtime template** in `asia-southeast1`), select Network **`acsm-colab-network`** and Subnetwork **`acsm-colab-subnet-sg`**, and click **Connect**.

---
## Step 1: Configure Parameters (`PROJECT_ID` & Singapore Region) and Clone Repository

In [ ]:
# @title 1. Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank) & Singapore Region (`asia-southeast1`)
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}
DATASET_ID = "acsm_bronze"
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
BIGLAKE_CONN_ID = f"acsm-biglake-iceberg-conn-{PROJECT_ID}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

# Load the native BigQuery SQL magic so all SQL cells run against $PROJECT_ID in $LOCATION
%load_ext google.cloud.bigquery

!gcloud config set project $PROJECT_ID
![ -d aeon-credit-gcp-workshop ] || git clone https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git
print(f"Active Project: {PROJECT_ID} | Region: {LOCATION} | Bucket: gs://{BUCKET_NAME}")


> **🔍 How to Verify Step 1 on GCP Console UI**
> 1. In the top Google Cloud Console bar, confirm your **`PROJECT_ID`** is selected in the Project Picker.
> 2. In the cell output above, confirm the repository cloned cleanly and `LOCATION=asia-southeast1` (Singapore) is active.
---
## Step 2: Create the Cloud Storage Landing Bucket in Singapore (`asia-southeast1`)

In [ ]:
# @title Step 2: Create Regional GCS Bucket in Singapore (asia-southeast1)
!gcloud storage buckets describe gs://{BUCKET_NAME} --project={PROJECT_ID} >/dev/null 2>&1 || \
  gcloud storage buckets create gs://{BUCKET_NAME} \
    --project={PROJECT_ID} \
    --location={LOCATION} \
    --uniform-bucket-level-access

!gcloud storage buckets describe gs://{BUCKET_NAME} --format="table(name,location,location_type,storage_class)"

> **🔍 How to Verify Step 2 on GCP Console UI (Cloud Storage Browser)**
> 1. Open **Cloud Storage $\rightarrow$ Buckets** in the GCP Console.
> 2. Verify bucket **`acsm-workshop-landing-${PROJECT_ID}`** shows **Location type**: `Region`, **Location**: `asia-southeast1 (Singapore)`, and **Public access**: `Not public`.
---
## Step 3: Copy Compressed Dataset Files (`.csv.gz`) from Repo to Singapore Bucket

In [ ]:
# @title Step 3: Upload the 7 GCP Compressed .csv.gz Files (6 Fact Tables + m3CIF; dimProduct lives in AWS S3)
!gcloud storage cp \
  aeon-credit-gcp-workshop/data/full_compressed/Fact_*.csv.gz \
  aeon-credit-gcp-workshop/data/full_compressed/m3CIF.csv.gz \
  gs://{BUCKET_NAME}/full_compressed/
!gcloud storage ls -l gs://{BUCKET_NAME}/full_compressed/


> **🔍 How to Verify Step 3 on GCP Console UI (Bucket Objects View)**
> 1. In **Cloud Storage $\rightarrow$ Buckets**, click **`acsm-workshop-landing-${PROJECT_ID}` $\rightarrow$ `full_compressed/`**.
> 2. Click **Refresh** and verify the 7 GCP `.csv.gz` files (`6` `Fact_*.csv.gz` tables + `m3CIF.csv.gz`) are listed in `asia-southeast1 (Singapore)` with `application/gzip` content type.
---
## Step 4: Storage Engine 1 — Run `CREATE TABLE` DDL for the 6 BigQuery Native Fact Tables (185 Column Descriptions)

> **📌 Architectural Note (3 Bronze Storage Engines across 8 Total Tables / 226 Columns)**:
> - **Storage Engine 1 (Step 4 & Step 5 — BigQuery Native Storage)**: Creates and loads the **6 Transactional Fact Tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection` — **`185` column descriptions**, **`1,233,284` rows**).
> - **Storage Engine 2 (Step 7 — GCP Lakehouse Open-Source Apache Iceberg)**: Creates **`acsm_bronze.m3CIF`** (**`28` column descriptions**, **`100,000` rows**) as a BigLake Managed Apache Iceberg table on GCS.
> - **Storage Engine 3 (Step 8 — Cross-Cloud AWS Glue Federated Apache Iceberg)**: Federates **`acsm_aws_bronze.dimProduct`** (**`13` columns**, **`65,000` rows**) directly from Amazon S3 + AWS Glue (`ap-southeast-1`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- =============================================================================
-- DEMO FLOW 2 (STEP 1 OF 2): Create 6 ACSM BigQuery Native Fact Tables with Table & Column Descriptions
-- Project: `<PROJECT_ID>` (e.g. `${PROJECT_ID}`) | Dataset: `acsm_bronze`
-- Location: `asia-southeast1` (Singapore)
-- Source Dictionary: `Mock Metadata.xlsx` (6 BigQuery Native Fact tables, 185 described columns; m3CIF is created as Iceberg in Step 7, dimProduct is federated from AWS Glue in Step 8)
-- =============================================================================

CREATE SCHEMA IF NOT EXISTS `acsm_bronze`
OPTIONS (
  location = "asia-southeast1",
  description = "AEON Credit Service Malaysia (ACSM) — 8 Core Tables (T1-T8) with Governed Metadata (Singapore Region)"
);

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `CIF_NO` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: numeric]"),
  `APPL_NO` INT64 OPTIONS(description = "PRODUCT application number [Source Data Type: varchar]"),
  `AGREE_NO` INT64 OPTIONS(description = "PRODUCT loan account number [Source Data Type: numeric]"),
  `APPL_DT` INT64 OPTIONS(description = "Easy payment application date [Source Data Type: numeric]"),
  `APPL_STS` STRING OPTIONS(description = "Easy payment application status [Source Data Type: varchar]"),
  `JUDGE_DT` INT64 OPTIONS(description = "Easy payment application decision date [Source Data Type: numeric]"),
  `SCORING_POINT` INT64 OPTIONS(description = "Easy payment Final Score for decision [Source Data Type: numeric]"),
  `SCORING_TYPE` INT64 OPTIONS(description = "Easy payment Final Score type [Source Data Type: numeric]"),
  `SCORING_RANK` STRING OPTIONS(description = "Easy payment Final score rank based on bucket [Source Data Type: varchar]"),
  `LOAN_CODE` INT64 OPTIONS(description = "Loan type [Source Data Type: varchar]"),
  `LOAN_TYP_ID` INT64 OPTIONS(description = "Loan subtype [Source Data Type: varchar]"),
  `LOAN_GRP` INT64 OPTIONS(description = "Loan group [Source Data Type: varchar]"),
  `AGENT_CODE1` INT64 OPTIONS(description = "Merchant group [Source Data Type: varchar]"),
  `AGENT_CODE2` INT64 OPTIONS(description = "Merchant subgroup [Source Data Type: varchar]"),
  `APPL_CHANNEL` STRING OPTIONS(description = "Application channel [Source Data Type: varchar]"),
  `REJECT_CODE` STRING OPTIONS(description = "Reject Code [Source Data Type: varchar]"),
  `FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment finance approved amount [Source Data Type: numeric]"),
  `FIN_PRFT_AMT` FLOAT64 OPTIONS(description = "Easy payment profit/interest amount [Source Data Type: numeric]"),
  `FIN_TOTAL_AMT` FLOAT64 OPTIONS(description = "Easy payment financing total amount [Source Data Type: numeric]"),
  `INST_AMT` FLOAT64 OPTIONS(description = "Easy payment installment amount [Source Data Type: numeric]"),
  `DEPOSIT` INT64 OPTIONS(description = "% for downpayment [Source Data Type: numeric]"),
  `INTEREST` FLOAT64 OPTIONS(description = "Easy payment interest [Source Data Type: decimal]"),
  `TOTAL_INST` INT64 OPTIONS(description = "Easy payment total installment months; loan tenure [Source Data Type: numeric]"),
  `JointIncome_FG` STRING OPTIONS(description = "Joint income yes/no flag [Source Data Type: varchar]"),
  `SCORE_DECISION` STRING OPTIONS(description = "Easy payment score decision: accept/decline [Source Data Type: char]"),
  `NetIncome` FLOAT64 OPTIONS(description = "Easy payment applicant net income [Source Data Type: numeric]"),
  `Income` FLOAT64 OPTIONS(description = "Easy payment applicant income [Source Data Type: numeric]"),
  `Age` INT64 OPTIONS(description = "Easy payment applicant age [Source Data Type: int]"),
  `HomeYear` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: numeric]"),
  `YearOfBusiness` INT64 OPTIONS(description = "Number of years of employment in current company [Source Data Type: numeric]"),
  `AnnualIncome` FLOAT64 OPTIONS(description = "Annual income [Source Data Type: numeric]"),
  `TOTAL_AEON_INST` INT64 OPTIONS(description = "Total installment for all Aeon products [Source Data Type: numeric]"),
  `TOTAL_AEON_OSB` FLOAT64 OPTIONS(description = "Total outstanding balance for all Aeon products [Source Data Type: numeric]"),
  `CUR_REPAY_RATIO` FLOAT64 OPTIONS(description = "Current repayment ratio [Source Data Type: decimal]"),
  `NEW_REPAY_RATIO` FLOAT64 OPTIONS(description = "New repayment ratio [Source Data Type: decimal]"),
  `NDI` FLOAT64 OPTIONS(description = "Net disposable income [Source Data Type: numeric]"),
  `CUR_DSR` FLOAT64 OPTIONS(description = "Current debt to service ratio [Source Data Type: decimal]"),
  `NEW_DSR` FLOAT64 OPTIONS(description = "New debt to service ratio [Source Data Type: decimal]"),
  `B_OtherIncome` FLOAT64 OPTIONS(description = "Other income [Source Data Type: numeric]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non-bank commitment [Source Data Type: numeric]"),
  `DEPENDANT` INT64 OPTIONS(description = "Children, spouse. [Source Data Type: numeric]"),
  `YEAR_MADE` INT64 OPTIONS(description = "Easy payment vehicle year made [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of Business [Source Data Type: varchar]"),
  `HomeOwner_flag` BOOL OPTIONS(description = "Ownership check of home [Source Data Type: varchar]"),
  `CIFState` STRING OPTIONS(description = "Customer address state as at application [Source Data Type: varchar]"),
  `Race` STRING OPTIONS(description = "Customer race as at application [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Customer gender as at application [Source Data Type: varchar]"),
  `Marital` STRING OPTIONS(description = "Customer marital status as at application [Source Data Type: varchar]"),
  `National` STRING OPTIONS(description = "Customer nationality as at application [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Customer occupation as at application [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Customer academic qualification as at application [Source Data Type: varchar]"),
  `B_AdvInstallPymt` FLOAT64 OPTIONS(description = "Easy payment advanced install payment [Source Data Type: numeric]"),
  `AdvInstallPymtBand` STRING OPTIONS(description = "Easy payment advanced installment payment band [Source Data Type: nvarchar]"),
  `JudgeWeek_FG` INT64 OPTIONS(description = "Judge week flag [Source Data Type: varchar]"),
  `Biometric_FG` BOOL OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `E-KYC` STRING OPTIONS(description = "E-KYC pass/fail [Source Data Type: varchar]"),
  `OTP` STRING OPTIONS(description = "OTP pass/fail [Source Data Type: varchar]"),
  `APPLY_FIN_AMT` FLOAT64 OPTIONS(description = "Easy payment applied financing amount [Source Data Type: numeric]"),
  `PreAssessment_Flag` BOOL OPTIONS(description = "Some customers qualify for pre-assessment [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for EP products. (Governed via Mock Metadata.xlsx | Sheet: T1 - Fact_EP_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Sales` (
  `TX_DT` FLOAT64 OPTIONS(description = "Sales Date"),
  `CIF_No` STRING OPTIONS(description = "Unique customer ID"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant"),
  `LDESC` STRING OPTIONS(description = "Spend Location"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount"),
  `TransCount` FLOAT64 OPTIONS(description = "Transaction Count")
) OPTIONS(description = "The confirmed sales log of the EP product. (Governed via Mock Metadata.xlsx | Sheet: T2 - Fact_EP_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_EP_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `First_INST_DT` INT64 OPTIONS(description = "First intallment date [Source Data Type: decimal]"),
  `Agree_No` INT64 OPTIONS(description = "Loan agreement ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `Current_Time_Payment` INT64 OPTIONS(description = "current installment period [Source Data Type: varchar]"),
  `Del_Sts` INT64 OPTIONS(description = "Lock Deliquency status [Source Data Type: varchar]"),
  `Collection_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score Point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score Grade [Source Data Type: varchar]"),
  `Sub_Code` STRING OPTIONS(description = "AKPK checker [Source Data Type: varchar]"),
  `Pay_in_Full` INT64 OPTIONS(description = "Account Status [Source Data Type: varchar]"),
  `Classification_Code` STRING OPTIONS(description = "Life Deliquency status [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "EP Multi Due Date [Source Data Type: varchar]"),
  `LoanTyp_ID` INT64 OPTIONS(description = "Loan product type indicator [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Principal Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Principal Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Principal Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Principal Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Principal Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Principal Unpaid count [Source Data Type: decimal]")
) OPTIONS(description = "The collection status snapshot for EP products as at closing period (Governed via Mock Metadata.xlsx | Sheet: T3 - Fact_EP_Collection)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Judge` (
  `Rcd_DT` DATE OPTIONS(description = "Data extraction date [Source Data Type: date]"),
  `Account_No` INT64 OPTIONS(description = "Card account number [Source Data Type: varchar]"),
  `Appl_ID` INT64 OPTIONS(description = "Credit card application ID [Source Data Type: varchar]"),
  `ApplSts_ID` STRING OPTIONS(description = "Credit card application status ID [Source Data Type: varchar]"),
  `CIF_ID` INT64 OPTIONS(description = "Credit card customer ID [Source Data Type: varchar]"),
  `CardTyp_ID` STRING OPTIONS(description = "Credit card type [Source Data Type: varchar]"),
  `CardBrand_ID` STRING OPTIONS(description = "Credit card brand [Source Data Type: varchar]"),
  `CardApplTyp_ID` STRING OPTIONS(description = "Principal/supplementary [Source Data Type: varchar]"),
  `ApplChnnl_ID` STRING OPTIONS(description = "Credit card application channel [Source Data Type: varchar]"),
  `Reject_ID` INT64 OPTIONS(description = "Credit card rejection reason ID [Source Data Type: varchar]"),
  `Decline_ID` INT64 OPTIONS(description = "Rejection reason [Source Data Type: varchar]"),
  `Agent_ID` INT64 OPTIONS(description = "Merchant ID [Source Data Type: varchar]"),
  `ScoreDecision_ID` STRING OPTIONS(description = "Credit card score decision category [Source Data Type: varchar]"),
  `ScoreRank_ID` STRING OPTIONS(description = "Credit card score rank [Source Data Type: varchar]"),
  `SysRcmmd_ID` STRING OPTIONS(description = "System recommended decision [Source Data Type: varchar]"),
  `Gender` STRING OPTIONS(description = "Gender [Source Data Type: varchar]"),
  `Age` INT64 OPTIONS(description = "Age [Source Data Type: int]"),
  `Race` STRING OPTIONS(description = "Race [Source Data Type: varchar]"),
  `Nationality` STRING OPTIONS(description = "Nationality short code [Source Data Type: varchar]"),
  `MaritalSts` STRING OPTIONS(description = "Marital Status [Source Data Type: varchar]"),
  `Academic` STRING OPTIONS(description = "Highest academic qualification [Source Data Type: varchar]"),
  `YrStay` INT64 OPTIONS(description = "Number of years living in current residence [Source Data Type: int]"),
  `HomeOwn` STRING OPTIONS(description = "Type of home ownership [Source Data Type: varchar]"),
  `Occupation` STRING OPTIONS(description = "Occupation [Source Data Type: varchar]"),
  `NOB` STRING OPTIONS(description = "Nature of business of customers employer [Source Data Type: varchar]"),
  `YrJob` INT64 OPTIONS(description = "Year in job [Source Data Type: int]"),
  `NetIncome` INT64 OPTIONS(description = "Net income [Source Data Type: int]"),
  `GrossIncome` INT64 OPTIONS(description = "Gross income [Source Data Type: int]"),
  `AnnualIncome` INT64 OPTIONS(description = "Annual income [Source Data Type: int]"),
  `AnnualIncomeTotLmt` INT64 OPTIONS(description = "Annual income total limit [Source Data Type: int]"),
  `CurrRepay` INT64 OPTIONS(description = "Current repayment amount [Source Data Type: int]"),
  `NewRepay` INT64 OPTIONS(description = "New repayment amount [Source Data Type: int]"),
  `RcmmdIntrst` FLOAT64 OPTIONS(description = "Recommended interest rate [Source Data Type: decimal]"),
  `NDI` INT64 OPTIONS(description = "Net Disposable Income [Source Data Type: int]"),
  `CurrDSR` INT64 OPTIONS(description = "Current DSR [Source Data Type: decimal]"),
  `NewDSR` INT64 OPTIONS(description = "New DSR [Source Data Type: decimal]"),
  `PaySlipTyp_ID` INT64 OPTIONS(description = "Payslip type [Source Data Type: varchar]"),
  `CardActivate_FG` BOOL OPTIONS(description = "Card activated [Source Data Type: varchar]"),
  `EmergencyCont_FG` BOOL OPTIONS(description = "Emergency contact provided [Source Data Type: varchar]"),
  `CardActivate_DT` INT64 OPTIONS(description = "Card activation date [Source Data Type: decimal]"),
  `Judge_DT` INT64 OPTIONS(description = "Decision date [Source Data Type: decimal]"),
  `Appl_DT` INT64 OPTIONS(description = "Application date [Source Data Type: decimal]"),
  `B_Limit` FLOAT64 OPTIONS(description = "Total limit [Source Data Type: decimal]"),
  `B_CrLimit` INT64 OPTIONS(description = "Credit limit [Source Data Type: decimal]"),
  `B_CashAdvLimit` FLOAT64 OPTIONS(description = "Cash advance limit [Source Data Type: decimal]"),
  `B_NonBankCommitment` FLOAT64 OPTIONS(description = "Non bank commitment [Source Data Type: decimal]"),
  `CIFState_ID` STRING OPTIONS(description = "State [Source Data Type: varchar]"),
  `Biometric_FG` STRING OPTIONS(description = "Biometric indicator [Source Data Type: varchar]"),
  `FinalDecline_ID` INT64 OPTIONS(description = "Decline ID [Source Data Type: varchar]"),
  `Final_Score` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_Score_Type` INT64 OPTIONS(description = "Credit card CTOS score [Source Data Type: decimal]"),
  `Final_ScoreDesc` STRING OPTIONS(description = "Credit card CTOS score description [Source Data Type: varchar]"),
  `ApplyCardBiz_ID` STRING OPTIONS(description = "Applied card ID [Source Data Type: varchar]"),
  `ProposedCardBiz_ID` STRING OPTIONS(description = "Proposed card ID [Source Data Type: varchar]")
) OPTIONS(description = "The current (daily full refreshed) application status (Approved or Rejected) for credit cards. (Governed via Mock Metadata.xlsx | Sheet: T4 - Fact_CC_Judge)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Sales` (
  `TX_DT` INT64 OPTIONS(description = "Sales Date [Source Data Type: decimal]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `TransactionCountry` STRING OPTIONS(description = "Ringgit Malaysia [Source Data Type: varchar]"),
  `TransactionCountryHigherLevel` STRING OPTIONS(description = "Local or Oversea [Source Data Type: varchar]"),
  `PriviledgeMerchantsGrp` STRING OPTIONS(description = "Member merchant [Source Data Type: varchar]"),
  `LDESC` STRING OPTIONS(description = "Spend Location [Source Data Type: varchar]"),
  `Sales_Type` STRING OPTIONS(description = "Cash Purchase or Cash Advance [Source Data Type: varchar]"),
  `Amount` FLOAT64 OPTIONS(description = "Transaction Amount [Source Data Type: decimal]"),
  `TransCount` INT64 OPTIONS(description = "Transaction Count [Source Data Type: decimal]")
) OPTIONS(description = "The actual spending & cash advance log on credit card (Governed via Mock Metadata.xlsx | Sheet: T5 - Fact_CC_Sales)");

CREATE OR REPLACE TABLE `acsm_bronze.Fact_CC_Collection` (
  `TX_DT` INT64 OPTIONS(description = "Reporting date [Source Data Type: decimal]"),
  `Account_No` INT64 OPTIONS(description = "CC account ID [Source Data Type: varchar]"),
  `CIF_No` INT64 OPTIONS(description = "Unique customer ID [Source Data Type: varchar]"),
  `DC_Sts` INT64 OPTIONS(description = "Lock deliquency status [Source Data Type: varchar]"),
  `Application_Branch` INT64 OPTIONS(description = "Branch ID [Source Data Type: varchar]"),
  `Score_Value` INT64 OPTIONS(description = "Collection Score point [Source Data Type: decimal]"),
  `Score_Grade` STRING OPTIONS(description = "Collection Score grade [Source Data Type: varchar]"),
  `MDD` INT64 OPTIONS(description = "CC Due Date [Source Data Type: varchar]"),
  `FinPlus_Code` STRING OPTIONS(description = "FinPlus (e-credit evaluation) tier [Source Data Type: varchar]"),
  `Billing_OSP` FLOAT64 OPTIONS(description = "Billing amount [Source Data Type: decimal]"),
  `Billing_Count` INT64 OPTIONS(description = "Billing count [Source Data Type: decimal]"),
  `Collection_OSP` FLOAT64 OPTIONS(description = "Collection amount [Source Data Type: decimal]"),
  `Collection_Count` INT64 OPTIONS(description = "Collection count [Source Data Type: decimal]"),
  `Unpaid_OSP` FLOAT64 OPTIONS(description = "Unpaid amount [Source Data Type: decimal]"),
  `Unpaid_Count` INT64 OPTIONS(description = "Unpaid count [Source Data Type: decimal]"),
  `Maintain_OSP` FLOAT64 OPTIONS(description = "Maintain amount [Source Data Type: decimal]"),
  `Maintain_Count` INT64 OPTIONS(description = "Maintain count [Source Data Type: decimal]")
) OPTIONS(description = "The billing and collection status snapshot for credit cards as at closing period (Governed via Mock Metadata.xlsx | Sheet: T6 - Fact_CC_Collection)");



> **🔍 How to Verify Step 4 on GCP Console UI (BigQuery Studio Explorer & Schema Tab)**
> 1. In the left **BigQuery Studio Explorer** pane (right beside this notebook!), expand **`${PROJECT_ID}` $\rightarrow$ `acsm_bronze`**.
> 2. Click on **`Fact_EP_Judge`** $\rightarrow$ select the **Schema** tab to verify all **60 columns** have their business **Description** populated from `Mock Metadata.xlsx`, and check the **Details** tab to confirm **Data location** is `asia-southeast1` and **Number of rows** is currently `0`.
---
## Step 5: Run Serverless `LOAD DATA OVERWRITE` Statement (**$0 Load Cost / `0 B Billed`**)
> **💡 Zero Compute Cost (`$0.00` / `0 Bytes Billed`)**: Batch loading data into BigQuery from Cloud Storage via the `LOAD DATA` SQL statement is **100% FREE (`$0.00`)** using BigQuery's shared batch slot pool ([BigQuery Pricing](https://cloud.google.com/bigquery/pricing#loading_data)). Because both the bucket and dataset are in **Singapore (`asia-southeast1`)**, there is **$0 network egress cost** and **100% of the 185 column descriptions across the 6 Native Fact tables** from Step 4 are preserved automatically.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- DEMO FLOW 2 (STEP 2 OF 2): Serverless SQL `LOAD DATA OVERWRITE` from GCS
-- Dynamically resolves `@@project_id` (`gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/`)
-- into the 6 pre-created BigQuery Native Fact tables in `acsm_bronze` (1,233,284 rows) while preserving all 185 column descriptions.
-- Zero compute provisioning required | $0 BigQuery batch load cost (0 B billed).
-- =============================================================================

DECLARE bucket_uri STRING DEFAULT CONCAT('gs://acsm-workshop-landing-', @@project_id, '/full_compressed');

-- 1. Load Fact_EP_Judge (140,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Judge.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 2. Load Fact_EP_Sales (119,859 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 3. Load Fact_EP_Collection (80,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_EP_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_EP_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 4. Load Fact_CC_Judge (227,500 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Judge`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Judge.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 5. Load Fact_CC_Sales (535,925 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Sales`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Sales.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);

-- 6. Load Fact_CC_Collection (130,000 rows)
EXECUTE IMMEDIATE FORMAT("""
LOAD DATA OVERWRITE `acsm_bronze.Fact_CC_Collection`
FROM FILES (
  format = 'CSV',
  uris = ['%s/Fact_CC_Collection.csv.gz'],
  skip_leading_rows = 1,
  allow_quoted_newlines = TRUE
);
""", bucket_uri);


---
## Step 6: Pure SQL Verification (`INFORMATION_SCHEMA` Audits — Zero Python)

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 1: Audit Row Counts, Table Descriptions & Described Columns (with Grand Total)
WITH table_stats AS (
  SELECT
    t.table_name,
    s.row_count,
    COUNT(c.column_name) AS total_columns,
    COUNTIF(c.description IS NOT NULL AND c.description != "") AS described_columns,
    REGEXP_REPLACE(COALESCE(opt.option_value, ""), r"^\"|\"$", "") AS table_description
  FROM `acsm_bronze.INFORMATION_SCHEMA.TABLES` t
  JOIN `acsm_bronze.__TABLES__` s
    ON t.table_name = s.table_id
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.TABLE_OPTIONS` opt
    ON t.table_name = opt.table_name AND opt.option_name = "description"
  LEFT JOIN `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` c
    ON t.table_name = c.table_name
  WHERE t.table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
  GROUP BY 1, 2, 5
)
SELECT
  table_name,
  row_count,
  total_columns,
  described_columns,
  ROUND(SAFE_DIVIDE(described_columns, total_columns) * 100, 1) AS coverage_pct,
  table_description
FROM table_stats
UNION ALL
SELECT
  "TOTAL (ALL 6 ACSM TABLES)" AS table_name,
  SUM(row_count) AS row_count,
  SUM(total_columns) AS total_columns,
  SUM(described_columns) AS described_columns,
  ROUND(SAFE_DIVIDE(SUM(described_columns), SUM(total_columns)) * 100, 1) AS coverage_pct,
  "100% Serverless Load Complete | 0 Bytes Billed ($0.00)" AS table_description
FROM table_stats
ORDER BY CASE WHEN STARTS_WITH(table_name, "TOTAL") THEN 2 ELSE 1 END, table_name;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 2: Prove $0 Load Cost (0 Bytes Billed) via BigQuery INFORMATION_SCHEMA.JOBS
SELECT
  job_id,
  statement_type,
  destination_table.table_id AS loaded_table,
  state,
  COALESCE(total_bytes_billed, 0) AS bytes_billed,
  "$0.00 (Free Shared Batch Pool)" AS ingestion_compute_cost,
  TIMESTAMP_DIFF(end_time, start_time, SECOND) AS duration_seconds
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE statement_type = "LOAD_DATA"
  AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
ORDER BY creation_time DESC
LIMIT 8;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION

-- Pure SQL Verification 3: Inspect Governed Column Descriptions from INFORMATION_SCHEMA
SELECT
  table_name,
  column_name,
  data_type,
  description
FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name IN ("Fact_EP_Judge", "Fact_EP_Sales", "Fact_EP_Collection", "Fact_CC_Judge", "Fact_CC_Sales", "Fact_CC_Collection", "m3CIF", "dimProduct")
ORDER BY table_name, column_name
LIMIT 25;


> **🔍 How to Verify Step 5 on GCP Console UI (4 Visual Checks in BigQuery Studio)**
> 1. **Verify `$0` Load Cost (`0 B Billed`)**: In the cell output above (or in BigQuery **Job history**), point out **`Total Bytes Billed: 0 B ($0.00 FREE Serverless Batch Load)`** in **`asia-southeast1`**.
> 2. **Verify Row Counts (`Details` Tab)**: In the left Explorer tree, click **`Fact_CC_Sales`** ($535,925$ rows) or **`Fact_EP_Judge`** ($140,000$ rows) $\rightarrow$ **Details** tab.
> 3. **Verify Loaded Records (`Preview` Tab — also $0 Cost)**: Click the **Preview** tab on any table to browse the records at zero query cost (`0 B billed`).
> 4. **Verify Preserved Column Descriptions (`Schema` Tab)**: Click the **Schema** tab to confirm all **185 column descriptions across the 6 Native Fact tables** remained intact after `LOAD DATA OVERWRITE`.

---
## Step 7: Storage Engine 2 — Create GCP Lakehouse Apache Iceberg Catalog & Write `m3CIF` (100,000 Rows) Using Apache Spark

Now that the six transactional fact tables are loaded into BigQuery Native Storage, we set up our second Bronze storage engine: an **Open-Source Apache Iceberg Lakehouse on Google Cloud Storage** managed through the **BigLake Iceberg REST Catalog** and populated directly by **Apache Spark (`PySpark`)**—without creating any temporary or staging table in BigQuery.

In this section, you will walk through six simple sub-steps—each in its own executable cell:
1. **Step 7.1**: Provision a dedicated regional Cloud Storage warehouse bucket in Singapore (`gs://acsm-lakehouse-iceberg-${PROJECT_ID}`) to store the open Apache Parquet data files and Iceberg metadata snapshots.
2. **Step 7.2**: Register the BigLake Iceberg REST Catalog (`acsm_gcp_lakehouse_catalog`) with vended credentials so both Apache Spark and BigQuery can access the Lakehouse catalog using a consistent name across projects.
3. **Step 7.3**: Grant Cloud Storage permissions to the Google-managed Service Account created for the Lakehouse catalog.
4. **Step 7.4**: Create the Bronze Lakehouse namespace (`acsm_gcp_bronze`) inside the catalog.
5. **Step 7.5**: Run **Apache Spark (`PySpark`)** connected to the BigLake Iceberg REST Catalog (`https://biglake.googleapis.com/iceberg/v1/restcatalog`) to read `m3CIF.csv.gz` (`100,000` rows) directly and write the open Apache Iceberg table (`acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`) onto Cloud Storage.
6. **Step 7.6**: Query the Spark-created Lakehouse Apache Iceberg table (`acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`) directly from **BigQuery Studio SQL** with zero data movement.


### Step 7.1 — Provision the Regional Cloud Storage Warehouse Bucket
This cell creates a dedicated Cloud Storage bucket in Singapore (`asia-southeast1`) with uniform bucket-level access and public access prevention enabled. This bucket acts as the physical object-storage warehouse where your open Apache Iceberg Parquet files and metadata manifests will reside.


In [ ]:
# @title Step 7.1: Create the Regional Cloud Storage Warehouse Bucket in Singapore
import os, subprocess
if "PROJECT_ID" not in globals() or not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
LOCATION = globals().get("LOCATION", "asia-southeast1")
LAKEHOUSE_BUCKET = f"acsm-lakehouse-iceberg-{PROJECT_ID}"
LAKEHOUSE_CATALOG = "acsm_gcp_lakehouse_catalog"
LAKEHOUSE_NAMESPACE = "acsm_gcp_bronze"

!gcloud services enable biglake.googleapis.com --project={PROJECT_ID} --quiet

!gcloud storage buckets describe gs://{LAKEHOUSE_BUCKET} --project={PROJECT_ID} >/dev/null 2>&1 || \
  gcloud storage buckets create gs://{LAKEHOUSE_BUCKET} \
    --project={PROJECT_ID} \
    --default-storage-class=STANDARD \
    --location={LOCATION} \
    --uniform-bucket-level-access \
    --public-access-prevention

!gcloud storage buckets describe gs://{LAKEHOUSE_BUCKET} --format="table(name,location,storage_class)"


### Step 7.2 — Register the BigLake Iceberg REST Catalog
This cell registers the BigLake Iceberg REST Catalog (`acsm_gcp_lakehouse_catalog`) in your project and links it to the Cloud Storage warehouse bucket created in Step 7.1. Using credential vending allows authorized query engines to securely access the underlying Iceberg files without sharing static storage keys.


In [ ]:
# @title Step 7.2: Register the BigLake Iceberg REST Catalog
LAKEHOUSE_BUCKET = f"acsm-lakehouse-iceberg-{PROJECT_ID}"
LAKEHOUSE_CATALOG = "acsm_gcp_lakehouse_catalog"

!gcloud biglake iceberg catalogs create {LAKEHOUSE_CATALOG} \
  --project={PROJECT_ID} \
  --catalog-type=biglake \
  --default-location=gs://{LAKEHOUSE_BUCKET} \
  --primary-location={LOCATION} \
  --credential-mode=vended-credentials 2>/dev/null || \
  gcloud alpha biglake iceberg catalogs create {LAKEHOUSE_CATALOG} \
    --project={PROJECT_ID} \
    --catalog-type=biglake \
    --default-location=gs://{LAKEHOUSE_BUCKET} \
    --primary-location={LOCATION} \
    --credential-mode=vended-credentials || true


### Step 7.3 — Authorize the Lakehouse Catalog Service Account on the Warehouse Bucket
When BigLake creates the Iceberg REST Catalog, it automatically provisions a dedicated Google-managed Service Account for that catalog. This cell looks up that Service Account identity and grants it Object User and Object Admin permissions on the Cloud Storage warehouse bucket so the catalog can manage Iceberg snapshots and Parquet files.


In [ ]:
# @title Step 7.3: Grant Cloud Storage Permissions to the Catalog Service Account
LAKEHOUSE_BUCKET = f"acsm-lakehouse-iceberg-{PROJECT_ID}"
LAKEHOUSE_CATALOG = "acsm_gcp_lakehouse_catalog"

sa_lines = !gcloud biglake iceberg catalogs describe {LAKEHOUSE_CATALOG} --project={PROJECT_ID} --format="value(biglake-service-account)" 2>/dev/null || gcloud alpha biglake iceberg catalogs describe {LAKEHOUSE_CATALOG} --project={PROJECT_ID} --format="value(biglake-service-account)"
LAKEHOUSE_SA = [line.strip() for line in sa_lines if "@" in line][-1]
print(f"✅ Lakehouse Catalog Service Account: {LAKEHOUSE_SA}")

!gcloud storage buckets add-iam-policy-binding gs://{LAKEHOUSE_BUCKET} \
  --member="serviceAccount:{LAKEHOUSE_SA}" \
  --role="roles/storage.objectUser" \
  --quiet

!gcloud storage buckets add-iam-policy-binding gs://{LAKEHOUSE_BUCKET} \
  --member="serviceAccount:{LAKEHOUSE_SA}" \
  --role="roles/storage.objectAdmin" \
  --quiet


### Step 7.4 — Create the Bronze Lakehouse Namespace
Just as BigQuery organizes native tables inside datasets, an Apache Iceberg catalog organizes tables inside namespaces. This cell creates the `acsm_gcp_bronze` namespace inside `acsm_gcp_lakehouse_catalog` so our GCP Lakehouse Bronze layer matches our naming convention across all storage engines.


In [ ]:
# @title Step 7.4: Create the Bronze Lakehouse Namespace
LAKEHOUSE_CATALOG = "acsm_gcp_lakehouse_catalog"
LAKEHOUSE_NAMESPACE = "acsm_gcp_bronze"

!gcloud biglake iceberg namespaces create {LAKEHOUSE_NAMESPACE} \
  --catalog={LAKEHOUSE_CATALOG} \
  --project={PROJECT_ID} 2>/dev/null || \
  gcloud alpha biglake iceberg namespaces create {LAKEHOUSE_NAMESPACE} \
    --catalog={LAKEHOUSE_CATALOG} \
    --project={PROJECT_ID} || true


### Step 7.5 — Read `m3CIF.csv.gz` & Write Directly into the Lakehouse Apache Iceberg Table Using Apache Spark (`PySpark`) & `GoogleAuthManager`
Instead of staging data inside BigQuery or manually managing OAuth tokens, this cell configures **Apache Spark (`PySpark`)** with **`org.apache.iceberg.gcp.auth.GoogleAuthManager`** and vended credentials connected to the **BigLake Iceberg REST Catalog (`https://biglake.googleapis.com/iceberg/v1/restcatalog`)**. Spark reads the 100,000 customer records directly from `m3CIF.csv.gz` and writes them as open Apache Parquet files and Iceberg metadata snapshots into `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF` on Cloud Storage.

In [ ]:
# @title Step 7.5: Read `m3CIF.csv.gz` & Write to `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF` Using PySpark + GoogleAuthManager
import os
import shutil
import subprocess
import urllib.request

if "PROJECT_ID" not in globals() or not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
LOCATION = globals().get("LOCATION", "asia-southeast1")
LAKEHOUSE_BUCKET = f"acsm-lakehouse-iceberg-{PROJECT_ID}"
LAKEHOUSE_CATALOG = "acsm_gcp_lakehouse_catalog"
LAKEHOUSE_NAMESPACE = "acsm_gcp_bronze"
LAKEHOUSE_TABLE = "m3CIF"
CATALOG_WAREHOUSE = f"bl://projects/{PROJECT_ID}/catalogs/{LAKEHOUSE_CATALOG}"

# 1. Ensure local CSV.GZ file is present
csv_path = "aeon-credit-gcp-workshop/data/full_compressed/m3CIF.csv.gz"
if not os.path.exists(csv_path):
    subprocess.run(["gcloud", "storage", "cp", f"gs://acsm-workshop-landing-{PROJECT_ID}/full_compressed/m3CIF.csv.gz", "/tmp/m3CIF.csv.gz"], check=True)
    csv_path = "/tmp/m3CIF.csv.gz"

# 2. Ensure Java 17/21 & clean SPARK_HOME so PySpark 4.0 / 3.5 Java Gateway starts cleanly
if not shutil.which("java"):
    print("☕ Installing OpenJDK 17 for local PySpark runtime...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)

for jvm_candidate in ["/usr/lib/jvm/java-17-openjdk-amd64", "/usr/lib/jvm/java-21-openjdk-amd64"]:
    if os.path.exists(jvm_candidate):
        os.environ["JAVA_HOME"] = jvm_candidate
        break
os.environ.pop("SPARK_HOME", None)

# Propagate Colab ADC file to Java child process if running in standard Google Colab
if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ and os.path.exists("/content/.config/application_default_credentials.json"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/.config/application_default_credentials.json"

try:
    import pyspark
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyspark"], check=True)
    import pyspark

from pyspark.sql import SparkSession

# 3. Download self-contained Apache Iceberg 1.10.1 Runtime & GCP Bundle JARs (includes GoogleAuthManager)
#    Note: Spark 4.0 uses Scala 2.13 (`4.0_2.13`), while Spark 3.5 uses Scala 2.12 (`3.5_2.12`).
spark_minor = ".".join(pyspark.__version__.split(".")[:2])
scala_suffix = "2.13" if spark_minor.startswith("4.") else "2.12"
iceberg_ver = "1.10.1"
jar_dir = "/tmp/iceberg_jars"
os.makedirs(jar_dir, exist_ok=True)

runtime_jar_name = f"iceberg-spark-runtime-{spark_minor}_{scala_suffix}-{iceberg_ver}.jar"
gcp_bundle_name = f"iceberg-gcp-bundle-{iceberg_ver}.jar"
runtime_jar_path = os.path.join(jar_dir, runtime_jar_name)
gcp_bundle_path = os.path.join(jar_dir, gcp_bundle_name)

maven_mirror = "https://storage-download.googleapis.com/maven-central/maven2/org/apache/iceberg"
for jar_name, artifact_id, dest_path in [
    (runtime_jar_name, f"iceberg-spark-runtime-{spark_minor}_{scala_suffix}", runtime_jar_path),
    (gcp_bundle_name, "iceberg-gcp-bundle", gcp_bundle_path),
]:
    if not os.path.exists(dest_path):
        url = f"{maven_mirror}/{artifact_id}/{iceberg_ver}/{jar_name}"
        print(f"⬇️ Downloading {jar_name}...")
        urllib.request.urlretrieve(url, dest_path)

# 4. Initialize SparkSession with GoogleAuthManager & BigLake Iceberg REST Catalog
if "spark" in globals() and globals()["spark"] is not None:
    try:
        globals()["spark"].stop()
    except Exception:
        pass

print(f"🚀 Starting PySpark ({pyspark.__version__}, Scala {scala_suffix}) with GoogleAuthManager connected to `{LAKEHOUSE_CATALOG}`...")
spark = (
    SparkSession.builder
    .appName("ACSM_PySpark_BigLake_Iceberg_Ingestion")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.jars", f"{runtime_jar_path},{gcp_bundle_path}")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.type", "rest")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.uri", "https://biglake.googleapis.com/iceberg/v1/restcatalog")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.warehouse", CATALOG_WAREHOUSE)
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.header.x-goog-user-project", PROJECT_ID)
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.rest.auth.type", "org.apache.iceberg.gcp.auth.GoogleAuthManager")
    .config(f"spark.sql.catalog.{LAKEHOUSE_CATALOG}.io-impl", "org.apache.iceberg.gcp.gcs.GCSFileIO")
    .getOrCreate()
)

# 5. Read m3CIF.csv.gz (100,000 rows) in Spark & write directly into the Lakehouse Iceberg table
print(f"📖 Reading `{csv_path}` (100,000 Customer Master rows) into Spark DataFrame...")
df_m3cif = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

full_iceberg_table = f"`{LAKEHOUSE_CATALOG}`.`{LAKEHOUSE_NAMESPACE}`.`{LAKEHOUSE_TABLE}`"
print(f"🧊 Writing 100,000 rows via PySpark directly into Iceberg table {full_iceberg_table} on gs://{LAKEHOUSE_BUCKET}...")
spark.sql(f"DROP TABLE IF EXISTS {full_iceberg_table}")
(
    df_m3cif.writeTo(f"{LAKEHOUSE_CATALOG}.{LAKEHOUSE_NAMESPACE}.{LAKEHOUSE_TABLE}")
    .tableProperty("write.format.default", "parquet")
    .tableProperty("gcp.biglake.table-management", "disabled")
    .createOrReplace()
)

spark_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {full_iceberg_table}").collect()[0]["cnt"]
print(f"✅ PySpark successfully wrote {spark_count:,} rows into `{LAKEHOUSE_CATALOG}.{LAKEHOUSE_NAMESPACE}.{LAKEHOUSE_TABLE}`!")

### Step 7.6 — Query the Spark-Created Lakehouse Apache Iceberg Table Directly from BigQuery SQL
Without copying or loading any data into BigQuery storage, run this BigQuery SQL cell to query the exact `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF` Apache Iceberg table that PySpark just wrote to Cloud Storage in Step 7.5.


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 7.6 (Pure BigQuery SQL): Query the PySpark-Created GCP Lakehouse Apache
-- Iceberg Table (`acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF` — 100,000 rows)
-- =============================================================================

-- Create a governed pass-through view in `acsm_bronze.m3CIF` for downstream Silver/Gold Medallion compatibility
DROP TABLE IF EXISTS `acsm_bronze.m3CIF`;
CREATE OR REPLACE VIEW `acsm_bronze.m3CIF`
OPTIONS(description = 'Pass-through view over PySpark-created GCP Lakehouse Iceberg table acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF')
AS SELECT * FROM `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`;

-- Verify 100,000 rows and inspect state distribution directly from the 4-part Lakehouse Iceberg table
SELECT
  'acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF' AS gcp_lakehouse_iceberg_table,
  COUNT(*) AS total_customer_rows,
  COUNT(DISTINCT State) AS distinct_malaysian_states,
  ROUND(AVG(B_GrossIncome), 2) AS avg_gross_income_myr
FROM `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`;


---
## Step 8: Storage Engine 3 — Connect Cross-Cloud AWS Glue Federated Apache Iceberg Table (`dimProduct` — 65,000 Rows)

Finally, we connect BigQuery Studio to our third Bronze storage engine: an **Apache Iceberg table stored in Amazon S3 and cataloged in AWS Glue Data Catalog in the AWS Singapore region (`ap-southeast-1`)**.

Instead of building cross-cloud ETL pipelines or copying data out of AWS S3, BigQuery federates directly with AWS Glue using passwordless OIDC Web Identity Federation (`AssumeRoleWithWebIdentity`). Furthermore, the AWS IAM role (`acsm-gcp-trust-role`) is locked down with a least-privilege policy (`acsm-federated-only-policy`) so that **only the `acsm_aws_bronze.dimProduct` Credit Card Product Master table (65,000 rows)** is exposed in the BigQuery Studio UI.

![Slide 6 — Cross-Cloud AWS Glue Federated Catalog Architecture Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/slide6_aws_glue_flow.png)


### Step 8.1 — Register the Cross-Cloud AWS Glue Federated Catalog in BigQuery
This cell registers the `acsm_aws_federated_catalog` in BigQuery (`asia-southeast1`) and links it to the AWS Glue Data Catalog in AWS Singapore (`ap-southeast-1`) using the dedicated AWS Web Identity role (`acsm-gcp-trust-role`).


In [ ]:
# @title Step 8.1: Register the Cross-Cloud AWS Glue Federated Catalog
AWS_ACCOUNT_ID = "<AWS_ACCOUNT_ID>"  # @param {type:"string"}
AWS_ROLE_NAME = "acsm-gcp-trust-role"  # @param {type:"string"}
AWS_REGION = "ap-southeast-1"  # @param {type:"string"}
FEDERATED_CATALOG_NAME = "acsm_aws_federated_catalog"

!gcloud alpha biglake iceberg catalogs create {FEDERATED_CATALOG_NAME} \
  --project={PROJECT_ID} \
  --primary-location={LOCATION} \
  --catalog-type=federated \
  --federated-catalog-type=glue \
  --refresh-interval=300s \
  --glue-warehouse={AWS_ACCOUNT_ID} \
  --glue-aws-region={AWS_REGION} \
  --glue-aws-role-arn=arn:aws:iam::{AWS_ACCOUNT_ID}:role/{AWS_ROLE_NAME} || true


### Step 8.2 — Retrieve Your Unique BigLake Service Account ID for AWS Trust Whitelisting
When BigQuery creates the federated catalog in Step 8.1, Google Cloud generates a unique 21-digit Service Account Subject ID for your project. Run this cell to display your unique Service Account ID, and paste it into the shared Workshop Tracker Google Sheet so the instructor can authorize your ID in the AWS IAM Role trust policy.


In [ ]:
# @title Step 8.2: Display Your Unique BigLake Service Account ID for the Workshop Tracker Sheet
!echo "================================================================================"
!echo -n "🔑 YOUR BIGLAKE SERVICE ACCOUNT ID : " && \
  gcloud alpha biglake iceberg catalogs describe {FEDERATED_CATALOG_NAME} \
    --project={PROJECT_ID} \
    --primary-location={LOCATION} \
    --format="value(biglake-service-account-id)"
!echo "📋 PASTE YOUR SA ID IN THIS SHEET  : https://docs.google.com/spreadsheets/d/1oWyKOTAXlzLTMKrNPw3xa9l4XwGCg84aF6aP_XtYQcc/edit?resourcekey=0-M6VZHHykIa1ul5ZL0y7U1g&gid=0#gid=0"
!echo "🛡️ AWS IAM ROLE TO TRUST YOUR SA   : arn:aws:iam::{AWS_ACCOUNT_ID}:role/{AWS_ROLE_NAME}"
!echo "🧊 AWS GLUE ICEBERG TABLE IN BQ UI : `{PROJECT_ID}.{FEDERATED_CATALOG_NAME}.acsm_aws_bronze.dimProduct`"
!echo "================================================================================"


### Step 8.3 — Run a Single Unified SQL Query Joining All Three Bronze Storage Engines
Once your Service Account ID is authorized in the AWS trust policy, BigQuery Studio acts as a single unified query engine across all three Bronze storage architectures. This SQL cell joins:
1. **Credit Card Sales Transactions (`Fact_CC_Sales`)** from **BigQuery Native Storage**
2. **Customer Master Profiles (`m3CIF`)** from the **GCP Lakehouse Apache Iceberg Catalog on Cloud Storage**
3. **Credit Card Product Master (`dimProduct`)** from the **Federated AWS Glue Apache Iceberg Catalog on Amazon S3**


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 8.3 (Pure BigQuery SQL): Query the Federated AWS Glue Apache Iceberg
-- Table (`acsm_aws_federated_catalog.acsm_aws_bronze.dimProduct` — 65,000 rows)
-- AND Join Across All 3 Bronze Storage Engines in a Single Query!
-- =============================================================================
SELECT
  p.Brand_Card_Type,
  p.Card_Status,
  c.State AS Malaysian_State,
  COUNT(DISTINCT s.Account_No) AS Active_Accounts,
  ROUND(SUM(s.Total_Sales_Amt), 2) AS Total_CC_Sales_MYR
FROM `acsm_bronze.Fact_CC_Sales` AS s                                              -- Engine 1: BigQuery Native Storage
INNER JOIN `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF` AS c                 -- Engine 2: GCP Lakehouse Iceberg REST Catalog (GCS)
  ON s.CIF_ID = c.CIF_ID
INNER JOIN `acsm_aws_federated_catalog.acsm_aws_bronze.dimProduct` AS p            -- Engine 3: AWS Glue Federated Iceberg Catalog (S3)
  ON CAST(s.Account_No AS STRING) = p.Account_No
GROUP BY 1, 2, 3
ORDER BY Total_CC_Sales_MYR DESC
LIMIT 15;
